# Train a Second Independent RNN — Disconnected-Network Control

Trains a second path-integrating RNN (Sorscher et al., NeurIPS 2019) from a **different random
seed**, sharing the **same place-cell targets** (`example_pc_centers.npy`), architecture, and
trajectory distribution as the pre-trained network. The *only* difference is the recurrent-weight
initialisation.

**Why:** provides a network with **zero cross-connections** to the original, for the specificity
control in Figure 1. Cross-network cell pairs share position drive but have no coupling, so
`ΔpR² = pR²(cell+pos) − pR²(pos)` should collapse to ~0 while raw `pR²(cell)` stays high —
demonstrating why ΔpR² (not `pR²(cell)`) is the right coupling measure.

**Cost (measured on this Mac, MPS):** ~850–975 ms/step at `Ng=4096` → ~14 h for 50k steps.
`Ng=2048` ≈ 4 h, `Ng=1024` ≈ 1.5 h. **Nothing is missing from the setup** — the model simply needs
~50–100k gradient steps to reach ~5 cm decoding; short runs look like they aren't learning because
loss only crosses the uniform-baseline (`log 512 ≈ 6.24`) after a few thousand steps.
Training is checkpointed and **resumable**: re-run the training cell to continue toward `TARGET_STEPS`.

In [ ]:
import sys, os, time, warnings
warnings.filterwarnings('ignore')
REPO_DIR = '/Users/harryclark/Documents/spatial-manifolds/GRID-PATTERN-FORMATION'
sys.path.insert(0, REPO_DIR)

import numpy as np
import torch
import matplotlib.pyplot as plt
from tqdm import tqdm

from place_cells          import PlaceCells
from trajectory_generator import TrajectoryGenerator
from model                import RNN
from scores               import GridScorer
from visualize            import compute_ratemaps
from utils                import generate_run_ID

DEVICE = 'mps'  if torch.backends.mps.is_available() else \
         'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', DEVICE, '| torch', torch.__version__)

## Config & build model

In [ ]:
# ── Config ────────────────────────────────────────────────────────────────────
SEED         = 1234     # different from the pretrained net -> independent weights
NG           = 4096     # 4096 matches pretrained (~14h/50k). 2048~=4h, 1024~=1.5h.
TARGET_STEPS = 50000    # total gradient steps to train toward

class Options: pass
options = Options()
options.Np = 512;              options.Ng = NG
options.sequence_length = 20;  options.batch_size = 200
options.learning_rate = 1e-4;  options.weight_decay = 1e-4
options.place_cell_rf = 0.12;  options.surround_scale = 2
options.RNN_type = 'RNN';      options.activation = 'relu'
options.DoG = True;            options.periodic = False
options.box_width = 2.2;       options.box_height = 2.2
options.device = DEVICE
options.run_ID = generate_run_ID(options)

SAVE_DIR = f'/Users/harryclark/Documents/spatial-manifolds/data/rnn_xgboost/second_net_seed{SEED}_Ng{NG}'
os.makedirs(SAVE_DIR, exist_ok=True)
CKPT = os.path.join(SAVE_DIR, 'ckpt.pth')
print('save dir:', SAVE_DIR)

# ── Build model with SHARED place-cell centers (same targets as net 1) ────────
torch.manual_seed(SEED); np.random.seed(SEED)
place_cells = PlaceCells(options)
place_cells.us = torch.tensor(
    np.load(os.path.join(REPO_DIR, 'models', 'example_pc_centers.npy')),
    dtype=torch.float32).to(DEVICE)

trajectory_generator = TrajectoryGenerator(options, place_cells)
model     = RNN(options, place_cells).to(DEVICE)
optimizer = torch.optim.Adam(model.parameters(), lr=options.learning_rate)

# ── Resume if a checkpoint exists ─────────────────────────────────────────────
history = {'step': 0, 'loss': [], 'err': []}
if os.path.isfile(CKPT):
    ck = torch.load(CKPT, map_location=DEVICE)
    model.load_state_dict(ck['model']); optimizer.load_state_dict(ck['optim'])
    history = ck['history']
    print(f"Resumed at step {history['step']}  "
          f"(err {np.mean(history['err'][-50:])*100:.1f} cm)")
else:
    print('Fresh model — no checkpoint yet.')

## Train (resumable — re-run this cell to continue)

In [ ]:
# Re-run to continue toward TARGET_STEPS. Checkpoints every SAVE_EVERY steps.
SAVE_EVERY = 1000
gen   = trajectory_generator.get_generator()
start = history['step']

if start >= TARGET_STEPS:
    print(f"Already at {start} >= TARGET_STEPS={TARGET_STEPS}. Raise TARGET_STEPS to train more.")
else:
    model.train()
    pbar = tqdm(range(start, TARGET_STEPS), initial=start, total=TARGET_STEPS)
    for step in pbar:
        inputs, pc_outputs, pos = next(gen)
        model.zero_grad()
        loss, err = model.compute_loss(inputs, pc_outputs, pos)
        loss.backward()
        optimizer.step()
        history['loss'].append(loss.item())
        history['err'].append(err.item())
        history['step'] = step + 1
        if step % 50 == 0:
            pbar.set_description(f"loss {loss.item():.3f} | err {err.item()*100:.1f} cm")
        if (step + 1) % SAVE_EVERY == 0 or (step + 1) == TARGET_STEPS:
            torch.save({'model': model.state_dict(), 'optim': optimizer.state_dict(),
                        'history': history}, CKPT)
    print(f"Done. step={history['step']}  "
          f"err={np.mean(history['err'][-50:])*100:.1f} cm  (target ~5 cm)")

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 3.2))
err_cm = np.array(history['err']) * 100
ax[0].plot(err_cm, lw=0.6, color='k')
ax[0].axhline(5, color='C1', ls='--', lw=1, label='~5 cm target')
ax[0].set_title('Decoding error'); ax[0].set_xlabel('step'); ax[0].set_ylabel('cm')
ax[0].legend(frameon=False)
ax[1].plot(history['loss'], lw=0.6, color='k')
ax[1].axhline(np.log(options.Np), color='C0', ls='--', lw=1, label='uniform baseline')
ax[1].set_title('Loss'); ax[1].set_xlabel('step'); ax[1].legend(frameon=False)
for a in ax: a.spines[['top', 'right']].set_visible(False)
plt.tight_layout(); plt.show()
if len(err_cm): print(f"last-50 mean err: {err_cm[-50:].mean():.1f} cm")

## Evaluate — decoding & grid representations

In [ ]:
# True vs decoded trajectories (mirror of inspect_model.ipynb)
model.eval()
with torch.no_grad():
    inputs, pos, pc_outputs = trajectory_generator.get_test_batch()
    pred_pos = place_cells.get_nearest_cell_pos(model.predict(inputs)).cpu()
pos = pos.cpu(); us = place_cells.us.cpu()

fig, ax = plt.subplots(figsize=(5, 5))
for i in range(5):
    ax.plot(pos[:, i, 0], pos[:, i, 1], c='k', lw=2, label='True' if i == 0 else None)
    ax.plot(pred_pos[:, i, 0], pred_pos[:, i, 1], '.-', c='C1',
            label='Decoded' if i == 0 else None)
ax.scatter(us[:, 0], us[:, 1], s=15, alpha=0.4, c='lightgrey')
ax.set_xlim(-options.box_width/2, options.box_width/2)
ax.set_ylim(-options.box_height/2, options.box_height/2)
ax.set_xticks([]); ax.set_yticks([]); ax.legend(frameon=False)
plt.title('2nd network — path integration'); plt.show()
model.train()

In [ ]:
# Rate maps + grid scores (did grid representations form?)
RES = 40; N_AVG = max(1, 1000 // options.sequence_length)
model.eval()
activations, rate_map, g, pos_rm = compute_ratemaps(
    model, trajectory_generator, options, res=RES, n_avg=N_AVG, Ng=options.Ng)

LORES = 20
_, rate_map_lo, _, _ = compute_ratemaps(
    model, trajectory_generator, options, res=LORES, n_avg=N_AVG, Ng=options.Ng)
coord_range = ((-options.box_width/2, options.box_width/2),
               (-options.box_height/2, options.box_height/2))
scorer = GridScorer(LORES, coord_range,
                    list(zip([0.2]*10, np.linspace(0.4, 1.0, 10).tolist())))
scores = np.array([scorer.get_scores(rm.reshape(LORES, LORES))[0]
                   for rm in tqdm(rate_map_lo)])
model.train()

valid = scores[~np.isnan(scores)]
print(f'grid score: median {np.median(valid):.2f} | frac > 0.3: {(valid > 0.3).mean():.2f} '
      f'| n > 0.3: {(valid > 0.3).sum()}')

order = np.flip(np.argsort(np.nan_to_num(scores, nan=-1)))
fig, axes = plt.subplots(2, 8, figsize=(14, 3.8))
for ax, idx in zip(axes.ravel(), order[:16]):
    ax.imshow(activations[idx], cmap='jet'); ax.axis('off')
    ax.set_title(f'{scores[idx]:.2f}', fontsize=7)
plt.suptitle('Top grid cells — 2nd network', fontsize=11)
plt.tight_layout(); plt.show()

plt.figure(figsize=(4, 3))
plt.hist(valid, bins=20, color='grey')
plt.xlabel('grid score'); plt.ylabel('count'); plt.title('Grid score distribution')
plt.tight_layout(); plt.show()

## Export weights for the pairwise-XGBoost pipeline

In [ ]:
# Export in the (w0,w1,w2,w3) format understood by RNN.set_weights, so this
# network drops straight into rnn_grid_cell_xgboost.ipynb via load_trained_weights.
w0 = model.encoder.weight.detach().cpu().numpy().T     # (Np, Ng)
w1 = model.RNN.weight_ih_l0.detach().cpu().numpy().T   # (2,  Ng)
w2 = model.RNN.weight_hh_l0.detach().cpu().numpy().T   # (Ng, Ng)  recurrent (transposed)
w3 = model.decoder.weight.detach().cpu().numpy().T     # (Ng, Np)
weights = np.array([w0, w1, w2, w3], dtype=object)
WPATH = os.path.join(SAVE_DIR, f'second_net_seed{SEED}_Ng{options.Ng}_weights.npy')
np.save(WPATH, weights, allow_pickle=True)
print('saved weights ->', WPATH)

# Ground-truth recurrent connectivity J: entry [i, j] is the weight from unit j -> unit i.
J = model.RNN.weight_hh_l0.detach().cpu().numpy()      # (Ng, Ng)
np.save(os.path.join(SAVE_DIR, 'J_recurrent.npy'), J)
print('J shape', J.shape, '-> saved J_recurrent.npy')

## How this feeds the Figure 1 disconnected-network control

In `rnn_grid_cell_xgboost.ipynb`:

1. **Load both networks.** Net 1 = `models/example_trained_weights.npy` (pretrained).
   Net 2 = the `..._weights.npy` saved above. Both use the *same* `example_pc_centers.npy`,
   so they encode the same place fields.
2. **Generate activations for each on the SAME trajectory batch** (`model.g(inputs)`), giving two
   `(T, Ng)` activation matrices whose only shared signal is position.
3. **Stack the two unit populations** and run the existing pairwise XGBoost assay
   (`pr2_pos`, `pr2_cell`, `pr2_full`) across the combined set.
4. **Ground-truth connectivity is block-diagonal:** within-net = each net's `J_recurrent.npy`,
   **cross-net = 0 by construction**.
5. **The control panel:** for cross-net pairs (true coupling = 0), `pR²(cell)` is high (shared
   position code) but `ΔpR² ≈ 0`; for within-net pairs, both are elevated. This is the specificity
   evidence that ΔpR² measures coupling rather than common input, and doubles as a check that the
   position baseline is rich enough to absorb the shared drive.